# VRPP + Lookahead — execucao interactiva

Este notebook e apenas uma **casca fina** sobre o pacote `vrpp_lookahead`.
Toda a logica (matriz ORS, instancia, lookahead, modelo Gurobi, relatorios)
vive em `src/vrpp_lookahead/` — aqui so se escolhe a configuracao e se corre.

| Passo | Funcao | Script equivalente |
|---|---|---|
| 1 | `gerar_matriz_ors(cfg)` | `scripts/01_gerar_matriz_ors.py` |
| 2 | `construir_instancia(cfg)` | `scripts/02_construir_instancia.py` |
| 3 | `correr(cfg)` | `scripts/03_correr_vrpp.py` |

Os parametros editaveis (**B, Q, R, C, OMEGA, MIP_GAP, TIME_LIMIT, MAX_ROTAS**)
estao em `config/instancia_491_C7.yaml` e podem ser alterados aqui em memoria
(celula 3) sem tocar no ficheiro.

In [ ]:
# 1 — bootstrap: por o pacote no path e carregar a configuracao
import sys, os
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
sys.path.insert(0, str(RAIZ / 'scripts'))

from _comum import carregar_dotenv
carregar_dotenv(RAIZ / '.env')          # chave ORS (so necessaria no passo 1)

from vrpp_lookahead import (Config, carregar_instancia, construir_instancia, correr,
                            diagnosticar, gerar_matriz_ors, resumo_instancia)

CFG = Config.desde_yaml(RAIZ / 'config' / 'instancia_491_C7.yaml')
CFG.resumo()

In [ ]:
# 2 — (opcional) passos 1 e 2. Salte-os se ja tiver matriz/instancia no disco.
# gerar_matriz_ors(CFG)        # ~1 pedido por bloco de origens; precisa de ORS_API_KEY
# construir_instancia(CFG)     # junta atributos + coordenadas + matriz ORS

INST = carregar_instancia(CFG)
resumo_instancia(INST)

In [ ]:
# 3 — parametros editaveis (sobrepoem o YAML apenas nesta sessao)
CFG.modelo.B = 16          # densidade [kg/m3]
CFG.modelo.Q = 3500        # capacidade do veiculo [kg]
CFG.modelo.R = 0.1625      # receita [euro/kg]
CFG.modelo.C = 1.0         # custo [euro/km]
CFG.modelo.OMEGA = 0.1     # custo fixo por veiculo [euro]
CFG.modelo.MAX_ROTAS = 2   # k <= MAX_ROTAS
CFG.modelo.MIP_GAP = 0.05  # 5%
CFG.modelo.TIME_LIMIT = 21600
CFG.validar()

# Se mudou B, recarregue a instancia (CAP_CONT depende de B):
INST = carregar_instancia(CFG)

DIAG = diagnosticar(INST, CFG)

In [ ]:
# 4 — correr o VRPP + Lookahead (mapas e Excel vao para results/<etiqueta>/)
KPIS = correr(CFG, diagnostico=DIAG)

In [ ]:
# 5 — resumo
import pandas as pd
pd.DataFrame(KPIS).T